# RedQueen — GDGoC AI Challenge 2026 | Kaggle Training Notebook

**Pipeline (10 cells):**
1. Cell 1 — Setup: install deps, clone repo
2. Cell 2 — Configure output directories
3. Cell 3 — Phase 0: History mining (optional, attach `history_game/` dataset)
4. Cell 4 — Generate BC data via GeniusRuleAgent rollouts (if no history attached)
5. Cell 5 — Phase 2: Behavioral Cloning (focal loss, 40 epochs)
6. Cell 6 — Phase 3: PPO Curriculum (7 stages, ent_coef 0.08→0.03)
7. Cell 7 — Phase 4: Self-play (optional, skip if time is tight)
8. Cell 8 — Export: ONNX (primary) + TorchScript (fallback)
9. Cell 9 — Prepare submission folder (3 files: `agent.py`, `model.onnx`, `model.pt`)
10. Cell 10 (LAST) — Zip both output folders for download

**Outputs:**
- `/kaggle/working/training_artifacts/` — all checkpoints, logs, BC dataset
- `/kaggle/working/submission/` — competition-ready 3-file folder
- `/kaggle/working/training_artifacts.zip`
- `/kaggle/working/submission.zip` (agent.py at root ✓)

> **Submission format rule**: `requirements.txt` is FORBIDDEN by the evaluator. Submission must contain only `agent.py`, `model.onnx`, `model.pt`.

In [ ]:
# ── Cell 1: Environment setup ────────────────────────────────────────────────
import os

# Suppress TF/XLA/gRPC C++ noise that fires when CUDA subprocesses start.
# Must be set BEFORE any subprocess spawns so worker processes inherit them.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"]      = "3"
os.environ["GRPC_VERBOSITY"]        = "ERROR"

import warnings
warnings.filterwarnings("ignore", message=".*shared layers.*", category=UserWarning)

import subprocess, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "redqueen"

# Install extra dependencies not available on Kaggle by default
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "sb3-contrib>=2.3.0",
    "stable-baselines3>=2.3.0",
    "gymnasium>=0.29.1",
    "onnx>=1.16.0",
    "onnxruntime>=1.18.0",
], check=True)

# Remove old unmaintained gym if pre-installed by Kaggle base image
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "gym"],
               capture_output=True)

# Clone or update repo
REPO_URL = "https://github.com/CryAndRRich/redqueen.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Setup complete. Repo at:", REPO_DIR)
print("Python:", sys.version)

In [ ]:
# ── Cell 2: Configure output directories ─────────────────────────────────────
from pathlib import Path

WORKING        = Path("/kaggle/working")
ARTIFACTS_DIR  = WORKING / "training_artifacts"
CKPT_DIR       = ARTIFACTS_DIR / "checkpoints"
DATA_DIR       = ARTIFACTS_DIR / "data"
LOGS_DIR       = ARTIFACTS_DIR / "logs"
PAST_AGENTS_DIR= CKPT_DIR / "past_agents"
SUBMISSION_DIR = WORKING / "submission"

for d in [CKPT_DIR, DATA_DIR, LOGS_DIR, PAST_AGENTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS_DIR)
print("Submission dir:", SUBMISSION_DIR)

In [ ]:
# ── Cell 3: Phase 0 — History mining (optional) ───────────────────────────────
#
# Attach history_game/ as a Kaggle dataset input and update HISTORY_DIR below.
# If not available, this cell is skipped and Cell 4 generates BC data from
# GeniusRuleAgent self-rollouts instead.
#
# Output: DATA_DIR/bc_dataset/  (directory of memmap .npy files)

from pathlib import Path

# Common Kaggle input locations for the history dataset
HISTORY_CANDIDATES = [
    Path("/kaggle/input/redqueen-history/history_game"),
    Path("/kaggle/input/bomberland-history/history_game"),
    REPO_DIR / "history_game",
]
HISTORY_DIR = next((p for p in HISTORY_CANDIDATES if p.exists()), None)

BC_DATASET = DATA_DIR / "bc_dataset"
USE_HISTORY_BC = HISTORY_DIR is not None and not (BC_DATASET / "_n.npy").exists()

if USE_HISTORY_BC:
    print(f"Found history_game at {HISTORY_DIR}")
    n_files = sum(1 for _ in HISTORY_DIR.rglob("*.json"))
    print(f"Match files: {n_files:,}")

    from src.training.history_parser import parse_history
    parse_history(
        history_dir=HISTORY_DIR,
        output_path=BC_DATASET,
        max_files=None,
        min_survival=120,
        min_bombs=5,
    )
elif (BC_DATASET / "_n.npy").exists():
    import numpy as np
    n = int(np.load(BC_DATASET / "_n.npy"))
    print(f"BC dataset already exists: {BC_DATASET}  ({n:,} transitions)")
else:
    print("No history_game found — Cell 4 will generate BC data from GeniusRuleAgent rollouts")

In [ ]:
# ── Cell 4: Generate BC data via GeniusRuleAgent self-rollout ─────────────────
# Runs only if Cell 3 found no history_game/. Skipped if BC dataset already exists.
#
# Streams transitions to disk in chunks — avoids accumulating GBs in RAM.
# Output: DATA_DIR/bc_dataset/  (same memmap format as Cell 3)

import numpy as np
import shutil
from pathlib import Path

BC_DATASET = DATA_DIR / "bc_dataset"

if (BC_DATASET / "_n.npy").exists():
    n = int(np.load(BC_DATASET / "_n.npy"))
    print(f"BC dataset exists ({BC_DATASET}, {n:,} transitions), skipping rollout")
else:
    print("Generating BC dataset from GeniusRuleAgent rollouts...")

    from engine.game import BomberEnv
    from agent import GeniusRuleAgent
    from src.utils.feature_extractor import extract_features, count_boxes
    from src.logic.action_masking import compute_action_mask

    N_GAMES    = 5_000   # ~650k transitions at 4 agents × 130 avg-survival steps
    MAX_STEPS  = 500
    CHUNK_SIZE = 50_000  # transitions per chunk file

    TMP_DIR = DATA_DIR / "_rollout_chunks"
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    chunk_idx = 0
    sp_buf, aux_buf, act_buf, mask_buf = [], [], [], []

    def flush_chunk():
        global chunk_idx, sp_buf, aux_buf, act_buf, mask_buf
        if not act_buf:
            return
        np.save(TMP_DIR / f"sp_{chunk_idx:04d}.npy",   np.stack(sp_buf).astype(np.float32))
        np.save(TMP_DIR / f"aux_{chunk_idx:04d}.npy",  np.stack(aux_buf).astype(np.float32))
        np.save(TMP_DIR / f"act_{chunk_idx:04d}.npy",  np.array(act_buf, dtype=np.int64))
        np.save(TMP_DIR / f"mask_{chunk_idx:04d}.npy", np.stack(mask_buf).astype(bool))
        chunk_idx += 1
        sp_buf, aux_buf, act_buf, mask_buf = [], [], [], []

    for game_seed in range(N_GAMES):
        env = BomberEnv(max_steps=MAX_STEPS, seed=game_seed)
        agents = [GeniusRuleAgent(i) for i in range(4)]
        obs = env.reset(seed=game_seed)
        initial_boxes = count_boxes(obs["map"])
        step = 0

        while True:
            actions_taken = [int(a.act(obs)) for a in agents]

            for aid in range(4):
                if int(obs["players"][aid][2]) == 0:
                    continue
                boxes_now = count_boxes(obs["map"])
                sp, aux = extract_features(
                    obs, aid,
                    step=step,
                    total_steps=MAX_STEPS,
                    initial_boxes=initial_boxes,
                    boxes_remaining=boxes_now,
                )
                mask = compute_action_mask(obs, aid)
                sp_buf.append(sp)
                aux_buf.append(aux)
                act_buf.append(actions_taken[aid])
                mask_buf.append(mask)

            if len(act_buf) >= CHUNK_SIZE:
                flush_chunk()

            next_obs, terminated, truncated = env.step(actions_taken)
            obs = next_obs
            step += 1
            if terminated or truncated:
                break

        if (game_seed + 1) % 500 == 0:
            print(f"  Game {game_seed+1}/{N_GAMES} | buffered {len(act_buf):,} | chunks {chunk_idx}")

    flush_chunk()

    # Merge chunks into a single memmap directory
    chunk_counts = [len(np.load(TMP_DIR / f"act_{i:04d}.npy")) for i in range(chunk_idx)]
    n_total = sum(chunk_counts)
    print(f"Merging {chunk_idx} chunks → {n_total:,} transitions")

    BC_DATASET.mkdir(parents=True, exist_ok=True)
    sp_mm   = np.memmap(BC_DATASET / "spatial.npy",      dtype="float32", mode="w+", shape=(n_total, 15, 13, 13))
    aux_mm  = np.memmap(BC_DATASET / "aux.npy",          dtype="float32", mode="w+", shape=(n_total, 7))
    act_mm  = np.memmap(BC_DATASET / "actions.npy",      dtype="int64",   mode="w+", shape=(n_total,))
    mask_mm = np.memmap(BC_DATASET / "action_masks.npy", dtype="bool",    mode="w+", shape=(n_total, 6))

    offset = 0
    for i in range(chunk_idx):
        sp_c   = np.load(TMP_DIR / f"sp_{i:04d}.npy")
        aux_c  = np.load(TMP_DIR / f"aux_{i:04d}.npy")
        act_c  = np.load(TMP_DIR / f"act_{i:04d}.npy")
        mask_c = np.load(TMP_DIR / f"mask_{i:04d}.npy")
        n_c = len(act_c)
        sp_mm[offset:offset+n_c]   = sp_c
        aux_mm[offset:offset+n_c]  = aux_c
        act_mm[offset:offset+n_c]  = act_c
        mask_mm[offset:offset+n_c] = mask_c
        offset += n_c

    del sp_mm, aux_mm, act_mm, mask_mm
    np.save(BC_DATASET / "_n.npy", np.array(n_total, dtype=np.int64))
    shutil.rmtree(TMP_DIR)
    print(f"Saved BC dataset → {BC_DATASET}/  ({n_total:,} transitions)")

In [ ]:
# ── Cell 5: Phase 2 — Behavioral Cloning ─────────────────────────────────────
# Trains BomberPolicyNet on (spatial, aux) → action with Focal Loss (γ=2).
# Focal loss handles class imbalance: PLACE_BOMB is ~15% of actions.
# Early stopping: patience=5 epochs with no val_loss improvement.
#
# Budget: ~17 min for 40 epochs on Kaggle T4.

from src.training.bc_trainer import train_bc

BC_DATASET = DATA_DIR / "bc_dataset"
assert (BC_DATASET / "_n.npy").exists(), f"BC dataset not found: {BC_DATASET}"

BC_EPOCHS = 40   # ~17 min on T4; early stopping may trigger before this

bc_best_ckpt = train_bc(
    dataset_path=BC_DATASET,
    output_dir=CKPT_DIR,
    epochs=BC_EPOCHS,
    batch_size=512,
    lr=3e-4,
    gamma_focal=2.0,
    device="auto",
    save_every=10,
)

print(f"BC best checkpoint: {bc_best_ckpt}")

In [ ]:
# ── Cell 6: Phase 3 — PPO curriculum training ─────────────────────────────────
#
# Curriculum stages (7 stages, each requires min 100k steps before advancing):
#   Stage 0: random          avg_rank ≤ 0.8  — dominate random agents
#   Stage 1: simple          avg_rank ≤ 1.2  — clearly beat simple agents
#   Stage 2: simple_smarter1 avg_rank ≤ 1.5  — BRIDGE: 1 Smarter + 2 Simple
#   Stage 3: smarter         avg_rank ≤ 1.8  — 2 Smarter + 1 Simple anchor
#   Stage 4: tactical        avg_rank ≤ 2.0  — 2 Tactical + 1 Smarter anchor
#   Stage 5: trapper         avg_rank ≤ 2.0  — 2 Trapper + 1 Tactical anchor
#   Stage 6: genius          avg_rank ≤ 2.0  — 2 Genius + 1 Tactical anchor
#
# avg_rank metric: 0=win (sole survivor), 3=died early. Lower is better.
# Evaluated every 50k steps over 200 episodes.
#
# Step-500 tie-break (BTC rule — 2026-05-26):
#   Surviving agents at step 500 are ranked among themselves by:
#     1. Kills  2. Boxes Destroyed  3. Items Collected  4. Bombs Placed
#   In evaluate(): draws assigned rank=(n_alive-1)/2.0 (expected under uniform tie-break).
#   The reward function already incentivises tie-break stats:
#     kills +2.0, boxes +0.4, items +0.3, bombs +0.003.
#
# Advancement rule (rolling mean):
#   Rolling mean of last 3 eval windows <= threshold AND >= 100k steps in stage.
#   Rolling mean replaces "3 consecutive": single unlucky windows no longer block advance.
#   (Observed: Stage 1 oscillated [1.38, 1.30, 1.69, 1.20, ...] never 3 consecutive < 1.2)
#
# ent_coef schedule (per stage, applied at stage start):
#   0.08 → 0.08 → 0.07 → 0.06 → 0.06 → 0.05 → 0.04
#   Stage 0+1 keep 0.08: ent_coef 0.06 caused entropy collapse → -0.35 nats at Stage 1.
#   Ref: Costa 2021 "32 Details of PPO"; Meishner et al. 2019 (arXiv:1911.04947).
#
# Other training settings:
#   - 20% random opponent mixing per training env (prevents co-adaptation)
#   - BC weight transfer: ALL 6 layer groups (spatial_enc, aux_enc, fusion,
#     policy_net[0], action_net, value_net) — not just feature extractor
#   - Stage-best checkpoint reload: after each stage, reload peak model (not final)
#   - If a stage's budget is exhausted, training stops and saves best checkpoint.
#     curriculum_all_passed=False → Cell 7 self-play is skipped automatically.
#
# Budget guide (Kaggle T4, N_ENVS=4):
#   BC 40 epochs               ≈  17 min
#   PPO 7 stages × 750k max   ≈ 315 min  (advances early when conditions are met)
#   Total                      ≈  ~5.5 h  (within 12h Kaggle limit)

from src.training.ppo_trainer import train_curriculum

PPO_STEPS_PER_STAGE = 750_000   # max steps per stage before stopping
N_ENVS = 4                       # parallel envs (tuned for Kaggle T4 VRAM)

# Returns (best_checkpoint_path, all_stages_passed).
# all_stages_passed=False if any stage failed — Cell 7 self-play is skipped automatically.
ppo_best_ckpt, curriculum_all_passed = train_curriculum(
    output_dir=CKPT_DIR,
    total_steps_per_stage=PPO_STEPS_PER_STAGE,
    n_envs=N_ENVS,
    init_from=bc_best_ckpt,
    device="auto",
)

print(f"PPO curriculum best: {ppo_best_ckpt}")
print(f"All stages passed:   {curriculum_all_passed}")

In [ ]:
# ── Cell 7 (optional): Phase 4 — Self-play ────────────────────────────────────
# Runs automatically only if curriculum (Cell 6) passed ALL 7 stages.
# Skipped automatically if any curriculum stage failed (budget exhausted).
# ~500k steps ≈ 50 min on Kaggle T4 at N_ENVS=4.
#
# Self-play trains against a rolling pool of the last 20 past snapshots,
# sampling recent ones more often (exponential-decay weights, α=0.9^age).

if curriculum_all_passed:
    print("All curriculum stages passed — starting self-play (Phase 4)...")
    from src.training.ppo_trainer import train_self_play

    ppo_best_ckpt = train_self_play(
        output_dir=CKPT_DIR,
        snapshot_dir=PAST_AGENTS_DIR,
        total_steps=500_000,
        n_envs=N_ENVS,
        init_from=ppo_best_ckpt,
        device="auto",
    )
    print(f"Self-play best: {ppo_best_ckpt}")
else:
    print("Self-play skipped — curriculum did not complete all 7 stages.")

In [ ]:
# ── Cell 8: Export ONNX + TorchScript ────────────────────────────────────────
# Produces two model files for the submission folder:
#   model.onnx — primary inference via onnxruntime (fastest, required)
#   model.pt   — TorchScript fallback (torch.jit.load, if onnxruntime unavailable)
#
# The ONNX export pads the variable-length bomb tensor to MAX_BOMBS=16
# so the graph has fixed input shapes (required for ONNX compatibility).

from src.utils.export_onnx import export_to_onnx, export_torchscript

# Use the best available checkpoint (PPO > BC)
best_ckpt = ppo_best_ckpt if "ppo_best_ckpt" in dir() else bc_best_ckpt
assert best_ckpt.exists(), f"No checkpoint found at {best_ckpt}"

onnx_path = CKPT_DIR / "model.onnx"
pt_path   = CKPT_DIR / "model.pt"

export_to_onnx(
    checkpoint_path=best_ckpt,
    output_path=onnx_path,
    opset=17,
    verify=True,
)

export_torchscript(
    checkpoint_path=best_ckpt,
    output_path=pt_path,
)

print(f"ONNX model:        {onnx_path}")
print(f"TorchScript model: {pt_path}")

In [ ]:
# ── Cell 9: Prepare submission folder ─────────────────────────────────────────
#
# Competition format — submission.zip must contain exactly these 3 files at root:
#   agent.py    ← MANDATORY entry point, must be at root (NOT inside a subfolder)
#   model.onnx  ← primary ONNX inference
#   model.pt    ← TorchScript fallback
#
# DO NOT include requirements.txt — the evaluator rejects it (requirements_txt_forbidden).

import shutil

shutil.copy2(REPO_DIR / "agent" / "agent.py", SUBMISSION_DIR / "agent.py")
shutil.copy2(onnx_path, SUBMISSION_DIR / "model.onnx")
shutil.copy2(pt_path,   SUBMISSION_DIR / "model.pt")

sub_files = list(SUBMISSION_DIR.iterdir())
print("Submission folder contents:")
for f in sorted(sub_files):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s}  {size_kb:.1f} KB")

assert (SUBMISSION_DIR / "agent.py").exists(),   "FATAL: agent.py missing from submission root!"
assert (SUBMISSION_DIR / "model.onnx").exists(), "FATAL: model.onnx missing!"
assert (SUBMISSION_DIR / "model.pt").exists(),   "FATAL: model.pt missing!"
assert not (SUBMISSION_DIR / "requirements.txt").exists(), \
    "FATAL: requirements.txt is FORBIDDEN — remove it!"
print("\nagent.py at root:         ✓")
print("model.onnx present:       ✓")
print("model.pt present:         ✓")
print("requirements.txt absent:  ✓")

In [ ]:
# ── Cell 10 (LAST): Zip both output folders ────────────────────────────────────
#
# After this cell completes, download from the "Output" tab:
#   /kaggle/working/training_artifacts.zip  — all checkpoints, logs, BC dataset
#   /kaggle/working/submission.zip          — upload this to the competition

import zipfile
from pathlib import Path

def zip_directory(source_dir: Path, output_zip: Path) -> None:
    """Zip source_dir into output_zip with all files at relative paths."""
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(source_dir.rglob("*")):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(source_dir))
    size_mb = output_zip.stat().st_size / (1024 * 1024)
    print(f"Created: {output_zip.name}  ({size_mb:.1f} MB)")


artifacts_zip  = WORKING / "training_artifacts.zip"
submission_zip = WORKING / "submission.zip"

zip_directory(ARTIFACTS_DIR, artifacts_zip)
zip_directory(SUBMISSION_DIR, submission_zip)

# Verify submission zip has agent.py at root (not inside a subfolder)
with zipfile.ZipFile(submission_zip, "r") as zf:
    names = zf.namelist()

print("\nFiles in submission.zip:")
for n in sorted(names):
    print(f"  {n}")

assert "agent.py" in names, "FATAL: agent.py not at root of submission.zip!"
print("\nagent.py at zip root: ✓")
print("\n=== Done! Download from the Output tab ===")
print(f"  → training_artifacts.zip  (all checkpoints)")
print(f"  → submission.zip          (upload this to the competition)")